# Load LR/DT surrogate explanations

In [ ]:
import json
import numpy as np
import pandas as pd
import os

## 1. Instance Loader

### 1.1 Data loader class

In [7]:
class AIDatasetLoader:
    def __init__(self, feature_values_df, metadata_df, AI_predictions_df):
        self.feature_values_df = feature_values_df
        self.metadata_df = metadata_df
        self.AI_predictions_df = AI_predictions_df

    def scale_value(self, value, min_val, max_val):
        if value < min_val:
            return 0
        elif value > max_val:
            return 1
        else:
            return (value - min_val) / (max_val - min_val)

    def scale_feature_values(self, instance_ids):
        scaled_features = []
        for instance_id in instance_ids:
            feature_row = self.feature_values_df[self.feature_values_df['instanceId'] == instance_id].iloc[0]
            app_id = feature_row['appId']
            app_metadata = self.metadata_df[self.metadata_df['appId'] == app_id]

            if app_metadata.empty:
                raise ValueError(f"No metadata found for appId: {app_id}")

            scaled_row = []
            i = 0
            while True:
                val_col = f'v{i}'
                min_col = f'v{i}_min'
                max_col = f'v{i}_max'

                if val_col not in feature_row.index or pd.isna(feature_row[val_col]):
                    break

                min_val = app_metadata[min_col].values[0] if min_col in app_metadata.columns else None
                max_val = app_metadata[max_col].values[0] if max_col in app_metadata.columns else None

                if pd.isna(min_val) and pd.isna(max_val):
                    scaled_row.append(feature_row[val_col])
                else:
                    value = feature_row[val_col]
                    scaled_row.append(self.scale_value(value, min_val, max_val))

                i += 1

            scaled_features.append(scaled_row)
        return scaled_features

    def load_instances(self, selected_ids, normalize=True):
        if not selected_ids:
            raise ValueError("selected_ids must be provided and cannot be empty.")

        if normalize:
            scaled_features = self.scale_feature_values(selected_ids)
        else:
            scaled_features = []
            for instance_id in selected_ids:
                feature_row = self.feature_values_df[self.feature_values_df['instanceId'] == instance_id].iloc[0]
                scaled_row = [feature_row[col] for col in feature_row.index if col.startswith('v')]
                scaled_features.append(scaled_row)

        AI_predictions = []
        for instance_id in selected_ids:
            pred_row = self.AI_predictions_df[self.AI_predictions_df['instanceId'] == instance_id]
            AI_predictions.append(pred_row.iloc[0]['pred'] if not pred_row.empty else None)

        return scaled_features, AI_predictions

    def filter_loader(self, condition):
        filtered_feature_values_df = self.feature_values_df[condition(self.feature_values_df)]
        filtered_metadata_df = self.metadata_df[condition(self.metadata_df)]
        filtered_predictions_df = self.AI_predictions_df[condition(self.AI_predictions_df)]

        return AIDatasetLoader(
            feature_values_df=filtered_feature_values_df,
            metadata_df=filtered_metadata_df,
            AI_predictions_df=filtered_predictions_df
        )


### 1.2 Load specific dataset

In [ ]:
current_dir = os.getcwd()
data_dir = os.path.join(current_dir, 'datasets')

In [9]:
file_values = os.path.join(data_dir, 'values.csv')
file_metadata = os.path.join(data_dir, 'metadata.csv')
file_prediction = os.path.join(data_dir, 'none.csv')

values_df = pd.read_csv(file_values)
metadata_df = pd.read_csv(file_metadata)
prediction_df = pd.read_csv(file_prediction)

In [10]:
# ✅ Which model to use per dataset
dataset_model_map = {
    "mushrooms": "mlp",
    "wine_quality": "mlp",
    "forest_cover": "xgboost",
    "adult": "xgboost",
}

# 🔧 Choose dataset here:
app_id = "wine_quality"  # 🔄 Change this line to switch datasets

# 🧠 Auto-configured values:
model_name = dataset_model_map[app_id]

In [11]:
ai_dataset_loader = AIDatasetLoader(
    feature_values_df=values_df,
    metadata_df=metadata_df,
    AI_predictions_df=prediction_df
)

# Apply filters based on expMethod and modelName
def ai_condition(df):
    condition = pd.Series([True] * len(df), index=df.index)
    if 'appId' in df.columns:
        condition &= (df['appId'] == app_id)
    if 'modelName' in df.columns:
        condition &= (df['modelName'] == model_name)
    return condition


ai_dataset_loader = ai_dataset_loader.filter_loader(ai_condition)

### 1.3 Example usage of Instance Loader

In [44]:
# feature names
print("Feature names:", ai_dataset_loader.feature_values_df.columns.tolist())

# Load instances without normalization
X, y_hat = ai_dataset_loader.load_instances([10], normalize=False)
print(X, y_hat)

Feature names: ['appId', 'instanceId', 'v0', 'v1', 'v2', 'v3', 'v4', 'y']
[[0.32, 17.0, 3.33, 0.78, 11.0]] [1]


## 2. Load Explanations

### 2.1 Load DT explanation

In [ ]:
class DecisionTreeInterpreter:
    def __init__(self, explanation_df, metadata_df, app_id, model_name):
        row = explanation_df[explanation_df['appId'] == app_id]
        if row.empty:
            raise ValueError(f"No decision tree explanation found for appId: {app_id}")
        self.explanation_row = row.iloc[0]

        row = metadata_df[metadata_df['appId'] == app_id]
        if row.empty:
            raise ValueError(f"No metadata found for appId: {app_id}")
        self.metadata_row = row.iloc[0]

        self.app_id = app_id
        self.model = model_name
        self.fidelity = self.explanation_row['fidelity']
        self.tree_structure = json.loads(self.explanation_row['tree_structure'])

        # ✅ Load and store class labels
        if "class_labels" in self.explanation_row:
            try:
                self.class_labels = json.loads(self.explanation_row["class_labels"])
            except Exception:
                self.class_labels = None
        else:
            self.class_labels = None

    def _format_feature(self, feature_key):
        if '=' in feature_key:
            base, cat_index = feature_key.split('=')
            v_index = base.replace('a', 'v')
            cat_col = f"{v_index}_{cat_index}"
            feat_name = self.metadata_row.get(base, base)
            cat_label = self.metadata_row.get(cat_col, f"Category {cat_index}")
            return f"{feat_name} = {cat_label}"
        else:
            return self.metadata_row.get(feature_key, feature_key)

    def print_tree(self, as_name=False):
        print(f"Decision Tree (appId={self.app_id}, model={self.model})")
        print(f"Fidelity: {self.fidelity:.4f}")
        self._print_node(0, 0, as_name)

    def _print_node(self, node_id, depth, as_name):
        node = next(n for n in self.tree_structure if n["node"] == node_id)
        prefix = "  " * depth
        if node["is_leaf"]:
            class_id = int(np.argmax(node["value"]))
            class_label = (
                self.class_labels[class_id]
                if self.class_labels and class_id < len(self.class_labels)
                else f"class {class_id}"
            )
            probs = np.round(node["value"], 4)
            print(f"{prefix}→ Predict {class_label} (probs: {probs})")
        else:
            feature = self._format_feature(node['feature']) if as_name else node['feature']
            print(f"{prefix}if {feature} <= {node['threshold']}:")
            self._print_node(node["left"], depth + 1, as_name)
            print(f"{prefix}else:")
            self._print_node(node["right"], depth + 1, as_name)

    def apply_to_instance(self, raw_input):
        """
        raw_input: numpy array of feature values (ordered a0, a1, a2, ...)
        Returns: predicted class label or class distribution
        """
        node = next(n for n in self.tree_structure if n["node"] == 0)
        while not node["is_leaf"]:
            feature_key = node["feature"]
            if '=' in feature_key:
                base, cat_idx = feature_key.split('=')
                col_idx = int(base[1:])
                val = 1.0 if int(raw_input[col_idx]) == int(cat_idx) else 0.0
            else:
                col_idx = int(feature_key[1:])
                val = raw_input[col_idx]

            if val <= node["threshold"]:
                node = next(n for n in self.tree_structure if n["node"] == node["left"])
            else:
                node = next(n for n in self.tree_structure if n["node"] == node["right"])

        # return full probs and optionally the class label
        return {
            "probs": node["value"],
            "class_index": int(np.argmax(node["value"])),
            "class_label": self.class_labels[int(np.argmax(node["value"]))] if self.class_labels else None
        }


In [14]:
dt_df = pd.read_csv(os.path.join(data_dir, 'decision_tree.csv'))

# Create decision tree loader
dt_exp = DecisionTreeInterpreter(dt_df, metadata_df, app_id, model_name)


In [46]:
dt_exp.print_tree(as_name=True)

Decision Tree (appId=wine_quality, model=mlp)
Fidelity: 0.8333
if Alcohol <= 10.575000286102295:
  if Vinegar Taint <= 0.32999999821186066:
    → Predict 1 (probs: [0.3333 0.6667])
  else:
    → Predict 0 (probs: [0.9608 0.0392])
else:
  if Vinegar Taint <= 0.5649999976158142:
    → Predict 1 (probs: [0.045 0.955])
  else:
    → Predict 0 (probs: [0.7812 0.2188])


In [23]:
instance_idx = 10


x, y_hat = ai_dataset_loader.load_instances([instance_idx], normalize=False)

# Print instance features, y_hat, and decision tree prediction
print(f"\nInstance ID: {instance_idx}")
print(f"Features: {x[0]}")
print(f"AI Prediction: {y_hat[0]}")
# Apply decision tree to the instance
dt_prediction = dt_exp.apply_to_instance(np.array(x[0]))
print(f"Decision Tree Prediction: {dt_prediction['class_label']} (probs: {dt_prediction['probs']})")


Instance ID: 10
Features: [0.32, 17.0, 3.33, 0.78, 11.0]
AI Prediction: 1
Decision Tree Prediction: 1 (probs: [0.04504504504504504, 0.954954954954955])


In [47]:
dt_exp.tree_structure

[{'node': 0,
  'feature': 'a4',
  'threshold': 10.575000286102295,
  'left': 1,
  'right': 4,
  'value': [0.5115384615384615, 0.48846153846153845],
  'is_leaf': False},
 {'node': 1,
  'feature': 'a0',
  'threshold': 0.32999999821186066,
  'left': 2,
  'right': 3,
  'value': [0.8803418803418803, 0.11965811965811966],
  'is_leaf': False},
 {'node': 2,
  'feature': None,
  'threshold': None,
  'left': None,
  'right': None,
  'value': [0.3333333333333333, 0.6666666666666666],
  'is_leaf': True},
 {'node': 3,
  'feature': None,
  'threshold': None,
  'left': None,
  'right': None,
  'value': [0.9607843137254902, 0.0392156862745098],
  'is_leaf': True},
 {'node': 4,
  'feature': 'a0',
  'threshold': 0.5649999976158142,
  'left': 5,
  'right': 6,
  'value': [0.2097902097902098, 0.7902097902097902],
  'is_leaf': False},
 {'node': 5,
  'feature': None,
  'threshold': None,
  'left': None,
  'right': None,
  'value': [0.04504504504504504, 0.954954954954955],
  'is_leaf': True},
 {'node': 6,
  '

### 2.2 Load LR explanation

In [25]:
from collections import OrderedDict

class LogisticRegressionInterpreter:
    def __init__(self, explanation_df, metadata_df, app_id, model_name):
        row = explanation_df[explanation_df['appId'] == app_id]
        if row.empty:
            raise ValueError(f"No logistic regression explanation found for appId: {app_id}")
        self.explanation_row = row.iloc[0]

        row = metadata_df[metadata_df['appId'] == app_id]
        if row.empty:
            raise ValueError(f"No metadata found for appId: {app_id}")
        self.metadata_row = row.iloc[0]

        self.app_id = app_id
        self.model = model_name
        self.fidelity = self.explanation_row['fidelity']
        self.intercept = self.explanation_row['intercept']

        # Keep coefficients ordered by feature index (a0, a1, a2=0, a2=1, ...)
        coef_keys = [k for k in self.explanation_row.index if k.startswith("coef_") and pd.notna(self.explanation_row[k])]
        coef_keys_sorted = sorted(coef_keys, key=lambda x: (int(x.split("_")[1][1:].split("=")[0]), x))
        self.coefficients = OrderedDict([
            (k.replace("coef_", ""), self.explanation_row[k]) for k in coef_keys_sorted
        ])

    def _format_feature(self, feature_key):
        if '=' in feature_key:
            base, cat_index = feature_key.split('=')
            v_index = base.replace('a', 'v')
            cat_col = f"{v_index}_{cat_index}"
            feat_name = self.metadata_row.get(base, base)
            cat_label = self.metadata_row.get(cat_col, f"Category {cat_index}")
            return f"{feat_name} = {cat_label}"
        else:
            return self.metadata_row.get(feature_key, feature_key)

    def print_model(self, as_name=False):
        print(f"Logistic Regression (appId={self.app_id}, model={self.model})")
        print(f"Fidelity: {self.fidelity:.4f}")
        print(f"Intercept: {self.intercept:.4f}")
        print("Coefficients:")
        for key, val in self.coefficients.items():
            name = self._format_feature(key) if as_name else key
            print(f"  {name:40} → {val:.4f}")

    def apply_to_instance(self, raw_input):
        """
        raw_input: numpy array of feature values (ordered as a0, a1, a2, ...)
        """
        z = self.intercept
        for key, coef in self.coefficients.items():
            if '=' in key:
                base, cat_idx = key.split('=')
                col_idx = int(base[1:])
                val = 1.0 if int(raw_input[col_idx]) == int(cat_idx) else 0.0
            else:
                col_idx = int(key[1:])
                val = raw_input[col_idx]
            z += coef * val
        return 1 / (1 + np.exp(-z))


In [26]:
logreg_df = pd.read_csv(os.path.join(data_dir, 'logistic_regression.csv'))

In [27]:
lr_exp = LogisticRegressionInterpreter(logreg_df, metadata_df, app_id, model_name)

# Print raw
lr_exp.print_model()

# Print with feature and category names
lr_exp.print_model(as_name=True)

Logistic Regression (appId=wine_quality, model=mlp)
Fidelity: 0.9750
Intercept: -1.8622
Coefficients:
  a0                                       → -3.8326
  a1                                       → -0.7534
  a2                                       → -0.4803
  a3                                       → 2.8920
  a4                                       → 4.5269
Logistic Regression (appId=wine_quality, model=mlp)
Fidelity: 0.9750
Intercept: -1.8622
Coefficients:
  Vinegar Taint                            → -3.8326
  SO2                                      → -0.7534
  pH                                       → -0.4803
  Sulphates                                → 2.8920
  Alcohol                                  → 4.5269


In [40]:
instance_idx = 10


x, y_hat = ai_dataset_loader.load_instances([instance_idx], normalize=True)

# Print instance features, y_hat, and decision tree prediction
print(f"\nInstance ID: {instance_idx}")
print(f"Features: {x[0]}")
print(f"AI Prediction: {y_hat[0]}")
# Apply decision tree to the instance
lr_prediction = lr_exp.apply_to_instance(np.array(x[0]))
print(f"Logistic Regression Prediction: {lr_prediction:.4f}")


Instance ID: 10
Features: [0.08771929824561402, 0.05934718100890212, 0.5294117647058826, 0.6739130434782609, 0.5454545454545455]
AI Prediction: 1
Logistic Regression Prediction: 0.8722
